## Load test and train data from drive!!

In [2]:
from google.colab import drive
drive.mount('/content/gdrive')

data_dir = '/content/gdrive/MyDrive/NLP Project/data/'

Mounted at /content/gdrive


In [3]:
import numpy as np
# Ensure data_dir is set correctly
data_dir = '/content/gdrive/MyDrive/NLP Project/data/'

x_train_path = data_dir + "X_train_simplified.txt"
x_test_path = data_dir + "X_test_simplified.txt"
y_train_path = data_dir + "y_train_simplified.txt"
y_test_path = data_dir + "y_test_simplified.txt"

# Load X_train (sentences) as text
with open(x_train_path, 'r') as f:
    X_train = f.read().splitlines()

# Load X_test (sentences) as text
with open(x_test_path, 'r') as f:
    X_test = f.read().splitlines()

# Load y_train (labels) as text and convert to int
with open(y_train_path, 'r') as f:
    y_train = [line.strip() for line in f.read().splitlines()]

# Load y_test (labels) as text and convert to int
with open(y_test_path, 'r') as f:
    y_test = [line.strip() for line in f.read().splitlines()]

print(f"Loaded X_train samples: {len(X_train)}")
print(f"Loaded X_test samples: {len(X_test)}")
print(f"Loaded y_train samples: {len(y_train)}")
print(f"Loaded y_test samples: {len(y_test)}")

# Define unique_labels here so it's available for subsequent cells
unique_labels = sorted(list(set(y_train)))

Loaded X_train samples: 15629
Loaded X_test samples: 3766
Loaded y_train samples: 15629
Loaded y_test samples: 3766


## NAIVE Baselines!!

Before implementing more complex models, we'll establish some baseline performance metrics.

### Baseline 1: Random Guessing

This baseline improves upon fully random guessing by weighting the probability of selecting each label based on its observed frequency in the training data. This provides a more realistic lower bound for performance.

In [4]:
from collections import Counter
import random
from sklearn.metrics import accuracy_score, f1_score

# Get the frequency of each label in the training set
label_counts = Counter(y_train)

# Get unique labels and their frequencies
possible_labels = list(label_counts.keys())
counts = list(label_counts.values())

# Calculate probabilities (weights) for each label
total_samples = sum(counts)
weights = [count / total_samples for count in counts]

# Generate weighted random predictions for the test set
weighted_random_predictions = random.choices(possible_labels, weights=weights, k=len(y_test))

# Calculate F1 score (using 'weighted' average for multiclass)
weighted_random_guessing_f1 = f1_score(y_test, weighted_random_predictions, average='weighted')
print(f"Weighted Random Guessing F1 Score: {weighted_random_guessing_f1:.4f}")

Weighted Random Guessing F1 Score: 0.3418


### Baseline 2 & 3: Feature Extraction (Word Length and Token Richness)

For the next two baselines, we'll extract simple linguistic features from each sentence:

*   Average Word Length
*   Type-Token Ratio (TTR)
*   [bruno note: add more baselines if you have any good simple ones]

We'll then train a simple `LogisticRegression` model using each of these features independently.

In [5]:
import nltk
from nltk.tokenize import word_tokenize
import numpy as np
from sklearn.linear_model import LogisticRegression

# Ensure punkt tokenizer is downloaded for word_tokenize
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

def extract_linguistic_features(sentence):
    words = word_tokenize(sentence.lower()) # Tokenize and convert to lowercase
    if not words:
        return 0.0, 0.0 # Handle empty sentences

    # Average Word Length
    avg_word_length = sum(len(word) for word in words) / len(words)

    # Type-Token Ratio (TTR)
    unique_words = set(words)
    ttr = len(unique_words) / len(words)

    return avg_word_length, ttr

# Extract features for training data
X_train_features = [extract_linguistic_features(s) for s in X_train]
X_train_word_length = np.array([f[0] for f in X_train_features]).reshape(-1, 1)
X_train_ttr = np.array([f[1] for f in X_train_features]).reshape(-1, 1)

# Extract features for testing data
X_test_features = [extract_linguistic_features(s) for s in X_test]
X_test_word_length = np.array([f[0] for f in X_test_features]).reshape(-1, 1)
X_test_ttr = np.array([f[1] for f in X_test_features]).reshape(-1, 1)

print(f"Shape of X_train_word_length: {X_train_word_length.shape}")
print(f"Shape of X_train_ttr: {X_train_ttr.shape}")

Shape of X_train_word_length: (15629, 1)
Shape of X_train_ttr: (15629, 1)


### Baseline 2: Average Word Length Classifier

Training a `LogisticRegression` model using only the average word length as a feature.

In [6]:
# Train a Logistic Regression model using average word length
lr_word_length = LogisticRegression(max_iter=1000, random_state=42)
lr_word_length.fit(X_train_word_length, y_train)

# Predict and evaluate
word_length_predictions = lr_word_length.predict(X_test_word_length)

# Calculate F1 score per class
word_length_f1_per_class = f1_score(y_test, word_length_predictions, average=None, labels=unique_labels)

# Calculate weighted average F1 score
word_length_f1_weighted = f1_score(y_test, word_length_predictions, average='weighted')

# Calculate macro average F1 score
word_length_f1_macro = f1_score(y_test, word_length_predictions, average='macro')

print("F1-scores per class (Average Word Length Baseline):")
for label, f1 in zip(unique_labels, word_length_f1_per_class):
    print(f"  {label}: {f1:.4f}")

print(f"\nAverage Word Length Baseline Weighted F1-score: {word_length_f1_weighted:.4f}")
print(f"Average Word Length Baseline Macro F1-score: {word_length_f1_macro:.4f}")

F1-scores per class (Average Word Length Baseline):
  A: 0.7335
  B: 0.4395
  C: 0.4167

Average Word Length Baseline Weighted F1-score: 0.5459
Average Word Length Baseline Macro F1-score: 0.5299


### Baseline 3: Type-Token Ratio Classifier

Training a `LogisticRegression` model using only the Type-Token Ratio as a feature.

[bruno note: guessing it would be a better baseline if we evaluated texts instead of sentences]

In [7]:
# Train a Logistic Regression model using Type-Token Ratio
lr_ttr = LogisticRegression(max_iter=1000, random_state=42)
lr_ttr.fit(X_train_ttr, y_train)

# Predict and evaluate
ttr_predictions = lr_ttr.predict(X_test_ttr)

# Calculate F1 score per class
ttr_f1_per_class = f1_score(y_test, ttr_predictions, average=None, labels=unique_labels)

# Calculate weighted average F1 score
ttr_f1_weighted = f1_score(y_test, ttr_predictions, average='weighted')

# Calculate macro average F1 score
ttr_f1_macro = f1_score(y_test, ttr_predictions, average='macro')

print("F1-scores per class (Type-Token Ratio Baseline):")
for label, f1 in zip(unique_labels, ttr_f1_per_class):
    print(f"  {label}: {f1:.4f}")

print(f"\nType-Token Ratio Baseline Weighted F1-score: {ttr_f1_weighted:.4f}")
print(f"Type-Token Ratio Baseline Macro F1-score: {ttr_f1_macro:.4f}")

F1-scores per class (Type-Token Ratio Baseline):
  A: 0.6738
  B: 0.2636
  C: 0.5064

Type-Token Ratio Baseline Weighted F1-score: 0.4903
Type-Token Ratio Baseline Macro F1-score: 0.4812
